# Baseline: Logistic Regression on FP_AMPL Features Only

**Purpose:** Verify whether the 3 hardware-extracted First Path Amplitude features  
(`FP_AMPL1`, `FP_AMPL2`, `FP_AMPL3`) are sufficient to trivially separate LOS from NLOS,  
making the 60-sample CIR sequence redundant for classification.

If a 0-hidden-layer model (logistic regression) scores ~99% using **only** the FP prior,  
it proves the dataset is trivially separable before any sequence processing begins.

In [3]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

## 1. Load & Prepare FP Features

Replicate the exact same FP normalization used in the neural network pipelines:  
`fp_i = FP_AMPL_i / max(RXPACC, 1) / 64.0`

In [4]:
CSV_PATH = "./dataset/channels/combined_uwb_dataset.csv"

df = pd.read_csv(CSV_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Label distribution:\n{df['Label'].value_counts().rename({0: 'LOS (0)', 1: 'NLOS (1)'}).to_string()}")

# Determine RXPACC column name
rxpacc_col = "RXPACC" if "RXPACC" in df.columns else "RX_PACC"

# Normalise FP features identically to neural network pipelines
rxpacc = df[rxpacc_col].values.astype(float)
rxpacc = np.maximum(rxpacc, 1.0)  # avoid division by zero

F = np.column_stack([
    df["FP_AMPL1"].values / rxpacc / 64.0,
    df["FP_AMPL2"].values / rxpacc / 64.0,
    df["FP_AMPL3"].values / rxpacc / 64.0,
]).astype(np.float32)

y = df["Label"].values.astype(np.float32)

print(f"\nFP feature matrix: {F.shape}")
print(f"Labels:            {y.shape}")
print(f"\nFP feature ranges:")
for i, name in enumerate(["FP_AMPL1", "FP_AMPL2", "FP_AMPL3"]):
    print(f"  {name}: [{F[:, i].min():.4f}, {F[:, i].max():.4f}]  mean={F[:, i].mean():.4f}")

Dataset shape: (3600, 1036)
Label distribution:
Label
NLOS (1)    1800
LOS (0)     1800

FP feature matrix: (3600, 3)
Labels:            (3600,)

FP feature ranges:
  FP_AMPL1: [0.0008, 1.0599]  mean=0.4689
  FP_AMPL2: [0.0155, 1.0595]  mean=0.5522
  FP_AMPL3: [0.0309, 1.0456]  mean=0.4697


## 2. Stratified 6-Fold Cross-Validation

In [5]:
K = 6
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=42)

fold_accuracies = []
fold_aucs = []

print(f"{'Fold':<6} {'Accuracy':>10} {'AUC':>10}")
print("-" * 28)

for fold, (train_idx, val_idx) in enumerate(skf.split(F, y)):
    F_train, F_val = F[train_idx], F[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # StandardScaler fitted on train fold only
    scaler = StandardScaler()
    F_train_s = scaler.fit_transform(F_train)
    F_val_s = scaler.transform(F_val)

    # Logistic Regression (no hidden layers, linear decision boundary)
    clf = LogisticRegression(max_iter=1000, solver="lbfgs", random_state=42)
    clf.fit(F_train_s, y_train)

    y_pred = clf.predict(F_val_s)
    y_prob = clf.predict_proba(F_val_s)[:, 1]

    acc = accuracy_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)

    fold_accuracies.append(acc)
    fold_aucs.append(auc)

    print(f"{fold + 1:<6} {acc:>10.4f} {auc:>10.4f}")

print("-" * 28)
print(f"{'Mean':<6} {np.mean(fold_accuracies):>10.4f} {np.mean(fold_aucs):>10.4f}")
print(f"{'Std':<6} {np.std(fold_accuracies):>10.4f} {np.std(fold_aucs):>10.4f}")

Fold     Accuracy        AUC
----------------------------
1          1.0000     1.0000
2          1.0000     1.0000
3          0.9967     0.9980
4          0.9983     0.9986
5          0.9983     1.0000
6          0.9983     0.9980
----------------------------
Mean       0.9986     0.9991
Std        0.0011     0.0009


## 3. Full-Data Classification Report

Train on all data and inspect learned coefficients + per-class metrics.

In [6]:
scaler_full = StandardScaler()
F_scaled = scaler_full.fit_transform(F)

clf_full = LogisticRegression(max_iter=1000, solver="lbfgs", random_state=42)
clf_full.fit(F_scaled, y)

y_pred_full = clf_full.predict(F_scaled)
y_prob_full = clf_full.predict_proba(F_scaled)[:, 1]

print("Full-data resubstitution metrics (for reference, NOT generalization):")
print(f"  Accuracy: {accuracy_score(y, y_pred_full):.4f}")
print(f"  AUC:      {roc_auc_score(y, y_prob_full):.4f}")
print()
print(classification_report(y, y_pred_full, target_names=["LOS", "NLOS"]))

print("Learned coefficients (standardised features):")
for name, coef in zip(["FP_AMPL1", "FP_AMPL2", "FP_AMPL3"], clf_full.coef_[0]):
    print(f"  {name}: {coef:+.4f}")
print(f"  Intercept: {clf_full.intercept_[0]:+.4f}")

Full-data resubstitution metrics (for reference, NOT generalization):
  Accuracy: 0.9992
  AUC:      0.9991

              precision    recall  f1-score   support

         LOS       1.00      1.00      1.00      1800
        NLOS       1.00      1.00      1.00      1800

    accuracy                           1.00      3600
   macro avg       1.00      1.00      1.00      3600
weighted avg       1.00      1.00      1.00      3600

Learned coefficients (standardised features):
  FP_AMPL1: -1.5689
  FP_AMPL2: -4.0636
  FP_AMPL3: -3.2797
  Intercept: -1.0341


## 4. Interpretation

If the mean CV accuracy is ≥ 95%, this confirms that the FP amplitudes alone  
are a near-perfect linear separator for LOS vs. NLOS in this dataset.  
Any neural network with access to `fp_features` can trivially learn this  
shortcut — the ~100% test accuracy of the complex models does **not**  
demonstrate that they learned CIR sequence dynamics.